# LSTM vs Transformer Encoder — AG News Text Classification

**Team:** 장지훈 (ETRI School, 02521122) · Deep Learning 2, 2026

이 노트북은 위에서 아래로 실행하면 전 과정을 재현합니다. 저장된 결과(`results_full.json`)·체크포인트(`*_full.pt`)가 있으면 로드하고, 없으면 학습합니다(MPS/CUDA에서 전체 ~30분).

**핵심 결과:** Transformer(test acc **0.910**) > LSTM(**0.833**). LSTM은 과적합·포화, Transformer는 안정적 확장. 과제는 bag-of-words(PE 꺼도 동일).

## 1. Setup

In [ ]:
import os, math, re, json, random, time
from collections import Counter
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset
from sklearn.metrics import f1_score, confusion_matrix
import matplotlib.pyplot as plt

SEED = 42
PAD_IDX, UNK_IDX, MAX_LEN = 0, 1, 128
torch.manual_seed(SEED); random.seed(SEED)
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 2. Dataset & Split
- Source: HuggingFace `ag_news` (단일 소스). 4 classes, label 0–3.
- Split: `train_test_split(test_size=0.1, seed=42)` → train 108k / val 12k / test 7.6k.
- test는 최종 평가 전용.

In [ ]:
ds = load_dataset("ag_news")
split = ds["train"].train_test_split(test_size=0.1, seed=SEED)
train_ds, val_ds, test_ds = split["train"], split["test"], ds["test"]
CLASSES = ds["train"].features["label"].names      # ['World','Sports','Business','Sci/Tech']
print("sizes  train/val/test:", len(train_ds), len(val_ds), len(test_ds))
print("classes:", CLASSES)
print("train class dist:", dict(sorted(Counter(train_ds["label"]).items())))

## 3. Preprocessing (두 모델 공통)
단어 단위 토큰화 → **train에서만** 어휘 구축(≤20k, min_freq 2, PAD/UNK) → 정수 인덱스 → 길이 128로 pad/truncate + mask.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9']+", text.lower())

def build_vocab(texts, max_size=20000, min_freq=2):
    c = Counter()
    for t in texts: c.update(tokenize(t))
    kept = [w for w, f in c.most_common() if f >= min_freq][:max_size-2]
    stoi = {"<pad>": PAD_IDX, "<unk>": UNK_IDX}
    for w in kept: stoi[w] = len(stoi)
    return stoi

def encode(text, stoi):
    return [stoi.get(w, UNK_IDX) for w in tokenize(text)]

def to_tensors(texts, labels, stoi, max_len=MAX_LEN):
    ids = torch.full((len(texts), max_len), PAD_IDX, dtype=torch.long)
    for i, t in enumerate(texts):
        s = encode(t, stoi)[:max_len]
        if s: ids[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    mask = ids != PAD_IDX
    return ids, mask, torch.tensor(labels, dtype=torch.long)

def masked_mean(h, mask):
    m = mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1).clamp(min=1)

stoi = build_vocab(train_ds["text"])               # vocab from TRAIN only (leakage 방지)
print("vocab size:", len(stoi))
def make_loader(d, bs, shuffle=False):
    return DataLoader(TensorDataset(*to_tensors(d["text"], d["label"], stoi)), batch_size=bs, shuffle=shuffle)
train_loader = make_loader(train_ds, 64, True)
val_loader   = make_loader(val_ds, 256)
test_loader  = make_loader(test_ds, 256)

## 4. Models
공유 골격(Embedding → Encoder → masked mean-pool → Linear)에서 **Encoder만** 교체. LSTM out_dim=256(bidirectional), Transformer out_dim=128(d_model).

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, encoder, vocab_size, emb=128, n_classes=4, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb, padding_idx=PAD_IDX)
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(encoder.out_dim, n_classes)
    def forward(self, ids, mask):
        x = self.embedding(ids)
        h = self.encoder(x, mask)
        return self.head(self.dropout(masked_mean(h, mask)))

class LSTMEncoder(nn.Module):
    def __init__(self, emb=128, hidden=128, layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(emb, hidden, num_layers=layers, bidirectional=True,
                            batch_first=True, dropout=dropout)
        self.out_dim = hidden * 2
    def forward(self, x, mask):
        out, _ = self.lstm(x); return out

class PositionalEncoding(nn.Module):
    def __init__(self, d_model=128, max_len=MAX_LEN):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))
        pe = torch.zeros(max_len, d_model); pe[:, 0::2] = torch.sin(pos*div); pe[:, 1::2] = torch.cos(pos*div)
        self.register_buffer("pe", pe)
    def forward(self, x): return x + self.pe[:x.size(1)]

class TransformerEncoder(nn.Module):
    def __init__(self, emb=128, heads=4, ff=256, layers=2, dropout=0.1, use_pe=True):
        super().__init__()
        self.use_pe = use_pe
        self.pos = PositionalEncoding(emb)
        layer = nn.TransformerEncoderLayer(emb, heads, ff, dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, layers)
        self.out_dim = emb
    def forward(self, x, mask):
        if self.use_pe: x = self.pos(x)
        return self.enc(x, src_key_padding_mask=~mask)

def make_encoder(name): return LSTMEncoder() if name == "lstm" else TransformerEncoder()
def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 5. Train / Eval

In [ ]:
def train_one_epoch(model, loader, opt, crit):
    model.train(); total = 0
    for ids, mask, y in loader:
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        opt.zero_grad(); loss = crit(model(ids, mask), y); loss.backward(); opt.step()
        total += loss.item() * len(y)
    return total / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, crit):
    model.eval(); total, preds, ys = 0, [], []
    for ids, mask, y in loader:
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        logits = model(ids, mask); total += crit(logits, y).item() * len(y)
        preds += logits.argmax(1).cpu().tolist(); ys += y.cpu().tolist()
    acc = sum(p == t for p, t in zip(preds, ys)) / len(ys)
    return total / len(loader.dataset), acc, f1_score(ys, preds, average="macro")

## 6. Main comparison (LSTM vs Transformer)
각 모델 8 epoch, **val로만 모델 선택**(best checkpoint). 저장된 결과가 있으면 로드.

In [ ]:
def train_model(name, epochs=8):
    torch.manual_seed(SEED)
    model = TextClassifier(make_encoder(name), len(stoi)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); crit = nn.CrossEntropyLoss()
    history, best, best_state = [], -1, None
    for ep in range(epochs):
        trl = train_one_epoch(model, train_loader, opt, crit)
        vll, vacc, vf1 = evaluate(model, val_loader, crit)
        history.append(dict(epoch=ep, train_loss=trl, val_loss=vll, val_acc=vacc, val_f1=vf1))
        if vf1 > best: best, best_state = vf1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"[{name}] ep{ep} train {trl:.3f} | val {vll:.3f} acc {vacc:.3f} f1 {vf1:.3f}")
    model.load_state_dict(best_state); torch.save(best_state, f"{name}_full.pt")
    return model, history

crit = nn.CrossEntropyLoss()
if os.path.exists("results_full.json"):
    results = json.load(open("results_full.json")); print("loaded results_full.json")
else:
    results = {}
    for name in ["lstm", "transformer"]:
        model, history = train_model(name)
        preds, ys, confs = [], [], []
        model.eval()
        with torch.no_grad():
            for ids, mask, y in test_loader:
                prob = torch.softmax(model(ids.to(device), mask.to(device)), 1)
                conf, pred = prob.max(1)
                preds += pred.cpu().tolist(); ys += y.tolist(); confs += conf.cpu().tolist()
        results[name] = dict(name=name, params=n_params(model), history=history,
            test_acc=sum(p == t for p, t in zip(preds, ys))/len(ys),
            test_f1=f1_score(ys, preds, average="macro"),
            confusion=confusion_matrix(ys, preds).tolist(),
            test_true=ys, test_pred=preds, test_conf=confs)
    json.dump(results, open("results_full.json", "w"))

print(f"\n{'model':12s}{'params':>11s}{'test_acc':>10s}{'test_f1':>9s}")
for n in ["lstm", "transformer"]:
    r = results[n]; print(f"{n:12s}{r['params']:>11,}{r['test_acc']:>10.4f}{r['test_f1']:>9.4f}")

## 7. Results — loss curves & confusion matrix

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, name in zip(ax, ["lstm", "transformer"]):
    h = results[name]["history"]; ep = [d["epoch"] for d in h]
    a.plot(ep, [d["train_loss"] for d in h], 'o-', label='train')
    a.plot(ep, [d["val_loss"] for d in h], 's-', label='val')
    a.set_title(name); a.set_xlabel('epoch'); a.set_ylabel('loss'); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("fig_loss_curves.png", dpi=130); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, name in zip(ax, ["lstm", "transformer"]):
    cm = np.array(results[name]["confusion"]); a.imshow(cm, cmap="Blues")
    a.set_xticks(range(4)); a.set_xticklabels(CLASSES, rotation=45, ha="right")
    a.set_yticks(range(4)); a.set_yticklabels(CLASSES); a.set_xlabel("pred"); a.set_ylabel("true")
    a.set_title(f"{name} acc {results[name]['test_acc']:.3f}")
    for i in range(4):
        for j in range(4):
            a.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                   color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout(); plt.savefig("fig_confusion.png", dpi=130); plt.show()

## 8. Ablation — data efficiency (5/25/50/100%)
통제: 어휘는 full-train 고정 재사용, subset은 stratified+seed, val/test 전체. 저장된 결과 있으면 로드.

In [ ]:
def stratified_subset(labels, frac, seed=SEED):
    rng = random.Random(seed); by = {c: [] for c in range(4)}
    for i, l in enumerate(labels): by[l].append(i)
    sel = []
    for c in range(4):
        idx = by[c][:]; rng.shuffle(idx); sel += idx[:int(len(idx)*frac)]
    return sel

if os.path.exists("ablation_results.json"):
    abl = json.load(open("ablation_results.json")); print("loaded ablation_results.json")
else:
    abl = {}
    for frac in [0.05, 0.25, 0.50]:
        sub = train_ds.select(stratified_subset(train_ds["label"], frac))
        tl = DataLoader(TensorDataset(*to_tensors(sub["text"], sub["label"], stoi)), batch_size=64, shuffle=True)
        for name in ["lstm", "transformer"]:
            torch.manual_seed(SEED)
            model = TextClassifier(make_encoder(name), len(stoi)).to(device)
            opt = torch.optim.Adam(model.parameters(), lr=1e-3)
            best, bs = -1, None
            for _ in range(8):
                train_one_epoch(model, tl, opt, crit)
                _, _, vf1 = evaluate(model, val_loader, crit)
                if vf1 > best: best, bs = vf1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            model.load_state_dict(bs); _, ta, tf = evaluate(model, test_loader, crit)
            abl[f"{name}_{frac}"] = dict(model=name, frac=frac, n=len(sub), test_acc=ta, test_f1=tf)
    for name in ["lstm", "transformer"]:
        abl[f"{name}_1.0"] = dict(model=name, frac=1.0, n=len(train_ds), test_acc=results[name]["test_acc"], test_f1=results[name]["test_f1"])
    json.dump(abl, open("ablation_results.json", "w"))

fr = [0.05, 0.25, 0.50, 1.0]; ns = [abl[f"lstm_{f}"]["n"] for f in fr]
plt.figure(figsize=(6.6, 4.6))
plt.plot(ns, [abl[f"lstm_{f}"]["test_acc"] for f in fr], 'o-', lw=2, label='LSTM')
plt.plot(ns, [abl[f"transformer_{f}"]["test_acc"] for f in fr], 's-', lw=2, label='Transformer')
plt.xscale('log'); plt.xlabel('# training samples (log)'); plt.ylabel('test accuracy')
plt.title('Data-efficiency ablation'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("fig_ablation.png", dpi=130); plt.show()
for f in fr:
    print(f"frac {f:.2f} (n={abl[f'lstm_{f}']['n']:6d}): LSTM {abl[f'lstm_{f}']['test_acc']:.3f} | TF {abl[f'transformer_{f}']['test_acc']:.3f}")

## 9. Failure analysis
test 예측을 공통오답 / LSTM-only / TF-only로 분류. 공통오답의 대부분이 Business↔Sci/Tech(의미 겹침).

In [ ]:
ys = results["lstm"]["test_true"]; pl = results["lstm"]["test_pred"]; pt = results["transformer"]["test_pred"]
cl = results["lstm"]["test_conf"]; ct = results["transformer"]["test_conf"]; texts = test_ds["text"]
N = len(ys)
both  = [i for i in range(N) if pl[i] != ys[i] and pt[i] != ys[i]]
lonly = [i for i in range(N) if pl[i] != ys[i] and pt[i] == ys[i]]
tonly = [i for i in range(N) if pt[i] != ys[i] and pl[i] == ys[i]]
print(f"both_wrong {len(both)} | LSTM-only {len(lonly)} | TF-only {len(tonly)}")
print(f"both_wrong 중 true=Business/Sci-Tech: {100*sum(1 for i in both if ys[i] in (2,3))/len(both):.0f}%\n")
def show(idxs, title, k=2):
    print(f"### {title}")
    for i in idxs[:k]:
        print(f"- {texts[i][:100]!r}")
        print(f"    true={CLASSES[ys[i]]} | LSTM={CLASSES[pl[i]]}({cl[i]:.2f}) | TF={CLASSES[pt[i]]}({ct[i]:.2f})")
show([i for i in both if ys[i] in (2,3)], "공통 오답 (Business/Sci-Tech)", 3)
show(lonly, "LSTM만 오답", 2); show(tonly, "TF만 오답", 2)

## 10. Extension — Transformer 메커니즘
**(A) PE on/off** (순서 의존도), **(B) gradient saliency**(두 모델 토큰 중요도), **(C) attention**(TF가 주목하는 토큰).

In [ ]:
# 학습된 모델 로드 (saliency/attention용)
def load_model(name):
    m = TextClassifier(make_encoder(name), len(stoi)); m.load_state_dict(torch.load(f"{name}_full.pt", map_location=device)); return m.to(device).eval()
lstm_model, tf_model = load_model("lstm"), load_model("transformer")

# (A) PE on/off  (no-PE = TransformerEncoder(use_pe=False))
if os.path.exists("extension_pe_results.json"):
    pe = json.load(open("extension_pe_results.json"))
else:
    torch.manual_seed(SEED)
    m = TextClassifier(TransformerEncoder(use_pe=False), len(stoi)).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3); best, bs = -1, None
    for _ in range(8):
        train_one_epoch(m, train_loader, opt, crit)
        _, _, vf1 = evaluate(m, val_loader, crit)
        if vf1 > best: best, bs = vf1, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
    m.load_state_dict(bs); _, acc, f1 = evaluate(m, test_loader, crit)
    pe = {"with_pe_acc": results["transformer"]["test_acc"], "no_pe_acc": acc, "no_pe_f1": f1,
          "drop": results["transformer"]["test_acc"] - acc}
    json.dump(pe, open("extension_pe_results.json", "w"))
print(f"(A) PE on/off: with-PE {pe['with_pe_acc']:.3f} vs no-PE {pe['no_pe_acc']:.3f} -> diff {pe['drop']:+.3f}  (≈0 => bag-of-words, 순서 불필요)")

# (B) gradient saliency
def saliency(model, text, k=6):
    toks = tokenize(text)[:MAX_LEN]
    ids = torch.tensor([[stoi.get(t, 1) for t in toks]]).to(device)
    mask = torch.ones_like(ids, dtype=torch.bool)
    emb = model.embedding(ids); emb.retain_grad()
    logits = model.head(masked_mean(model.encoder(emb, mask), mask)); pred = logits.argmax(1).item()
    model.zero_grad(); logits[0, pred].backward()
    sal = emb.grad[0].norm(dim=-1); sal = (sal/sal.sum()).tolist()
    return pred, sorted(zip(toks, sal), key=lambda x: -x[1])[:k]

# (C) attention (마지막 layer, forward hook)
def attention_top(text, k=6):
    store = {}
    h = tf_model.encoder.enc.layers[-1].self_attn.register_forward_pre_hook(
        lambda mod, a, kw: store.__setitem__('ak', (a, kw)), with_kwargs=True)
    toks = tokenize(text)[:MAX_LEN]
    ids = torch.tensor([[stoi.get(t, 1) for t in toks]]).to(device)
    mask = torch.ones_like(ids, dtype=torch.bool)
    with torch.no_grad():
        pred = tf_model(ids, mask).argmax(1).item()
        a, kw = store['ak']
        _, attn = tf_model.encoder.enc.layers[-1].self_attn(*a, **{**kw, 'need_weights': True, 'average_attn_weights': True})
    h.remove()
    recv = attn[0].mean(0)
    return pred, sorted(zip(toks, recv.tolist()), key=lambda x: -x[1])[:k]

ex = next(test_ds[i]["text"] for i in range(len(test_ds)) if test_ds[i]["label"] == 1)  # 실제 test 첫 Sports 기사
print("\n예시(true=Sports):", ex[:80], "...")
for nm, m in [("LSTM", lstm_model), ("TF", tf_model)]:
    pr, top = saliency(m, ex)
    print(f"  [{nm}] saliency pred={CLASSES[pr]}: " + ", ".join(f"{t}({s:.2f})" for t, s in top))
pr, top = attention_top(ex)
print(f"  [TF] attention pred={CLASSES[pr]}: " + ", ".join(f"{t}({s:.2f})" for t, s in top))

## 11. Conclusion
- **Transformer가 우위** (test acc 0.910 vs 0.833, 비슷한 파라미터 규모에서).
- **이유 = 일반화 차이**: LSTM은 심한 과적합(val_loss 폭증)·~25%에서 데이터 포화. Transformer는 안정적으로 확장.
- **혼동**: 두 모델 다 Business↔Sci/Tech(의미 겹침)에서 최다 오류. LSTM은 쉬운 사례도 흘림.
- **메커니즘**: PE 꺼도 정확도 동일 → AG News는 bag-of-words(순서 불필요). Transformer는 주제 키워드에 robust하게 주목, LSTM은 튀는 토큰에 취약.
- **데이터 효율**: "Transformer가 data-hungry"라는 가설은 반증 — 전 구간 Transformer 우위.
- **한계/후속**: 단일 seed(다중 seed 필요), LSTM 정규화(dropout↑) 시 격차 변화 확인, 순서가 중요한 과제에서 PE 효과 재검증.